# 3. 变分原理与优化方法

本节介绍变分蒙特卡洛方法的理论基础——变分原理，以及用于优化神经网络量子态的各种方法。

## 3.1 变分原理

变分原理是量子力学中的基本原理之一，它为我们提供了一种寻找系统基态能量的方法。在变分蒙特卡洛方法中，我们使用变分原理来优化神经网络量子态的参数。

### 变分原理的数学表述

对于任意归一化的试探波函数$|\psi(\mathbf{\theta})\rangle$，其能量期望值总是大于或等于系统的真实基态能量$E_0$：

$$E(\mathbf{\theta}) = \frac{\langle \psi(\mathbf{\theta}) | \hat{H} | \psi(\mathbf{\theta}) \rangle}{\langle \psi(\mathbf{\theta}) | \psi(\mathbf{\theta}) \rangle} \geq E_0$$

其中$\hat{H}$是系统的哈密顿量，$\mathbf{\theta}$是试探波函数的参数。

### 变分原理的物理意义

1. **能量上界**：任何试探波函数给出的能量期望值都是真实基态能量的上界
2. **优化方向**：通过最小化能量期望值，我们可以逼近真实基态
3. **参数依赖**：能量期望值是参数的函数，可以通过优化参数来降低能量

### 变分原理在蒙特卡洛方法中的应用

在变分蒙特卡洛方法中，我们使用蒙特卡洛采样来计算能量期望值：

$$E(\mathbf{\theta}) = \frac{\int |\psi(\mathbf{x}; \mathbf{\theta})|^2 E_L(\mathbf{x}; \mathbf{\theta}) d\mathbf{x}}{\int |\psi(\mathbf{x}; \mathbf{\theta})|^2 d\mathbf{x}} \approx \frac{1}{N} \sum_{i=1}^{N} E_L(\mathbf{x}_i; \mathbf{\theta})$$

其中$\mathbf{x}_i$是从概率分布$|\psi(\mathbf{x}; \mathbf{\theta})|^2$中采样的组态，$E_L(\mathbf{x}; \mathbf{\theta}) = \frac{\hat{H}\psi(\mathbf{x}; \mathbf{\theta})}{\psi(\mathbf{x}; \mathbf{\theta})}$是局域能量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set style for plots
plt.style.use('seaborn-v0_8')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define a simple RBM wave function for demonstration
class SimpleRBM(nn.Module):
    def __init__(self, n_visible, n_hidden):
        super(SimpleRBM, self).__init__()
        self.n_visible = n_visible
        self.n_hidden = n_hidden
        
        # Parameters
        self.a = nn.Parameter(torch.randn(n_visible))  # Visible bias
        self.b = nn.Parameter(torch.randn(n_hidden))  # Hidden bias
        self.W = nn.Parameter(torch.randn(n_visible, n_hidden))  # Weights
        
    def forward(self, x):
        # Visible layer contribution
        visible_term = torch.sum(x * self.a, dim=1)
        
        # Hidden layer contribution
        hidden_term = torch.sum(
            torch.log(2 * torch.cosh(torch.matmul(x, self.W) + self.b)),
            dim=1
        )
        
        return visible_term + hidden_term
    
    def log_psi(self, x):
        return self.forward(x)
    
    def psi(self, x):
        return torch.exp(self.log_psi(x))

# Create a simple RBM for demonstration
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Generate random spin configurations
batch_size = 100
spins = torch.randint(0, 2, (batch_size, n_spins)).float() * 2 - 1  # Convert to +/- 1
spins = spins.to(device)

# Compute wave function
log_psi = rbm.log_psi(spins)
psi = rbm.psi(spins)

print(f"Generated {batch_size} random spin configurations")
print(f"Shape of spin configurations: {spins.shape}")
print(f"Shape of log wave function: {log_psi.shape}")
print(f"Shape of wave function: {psi.shape}")

## 3.2 能量梯度计算

在变分蒙特卡洛方法中，我们需要计算能量关于参数的梯度，以便使用梯度下降等优化方法来最小化能量。能量梯度的计算是变分蒙特卡洛方法的核心步骤之一。

### 能量梯度的数学表达式

能量关于参数$\theta_k$的梯度可以表示为：

$$\frac{\partial E}{\partial \theta_k} = 2 \text{Re} \left[ \langle E_L \frac{\partial \log \psi^*}{\partial \theta_k} \rangle - \langle E_L \rangle \langle \frac{\partial \log \psi^*}{\partial \theta_k} \rangle \right]$$

其中$E_L$是局域能量，$\frac{\partial \log \psi^*}{\partial \theta_k}$是波函数对数关于参数的梯度。

对于实值波函数，上式简化为：

$$\frac{\partial E}{\partial \theta_k} = 2 \left[ \langle E_L \frac{\partial \log \psi}{\partial \theta_k} \rangle - \langle E_L \rangle \langle \frac{\partial \log \psi}{\partial \theta_k} \rangle \right]$$

### 梯度计算的物理意义

1. **能量-梯度关联**：能量梯度与局域能量和波函数梯度的关联有关
2. **中心化处理**：梯度计算需要减去平均值，以确保正确的优化方向
3. **统计估计**：梯度可以通过蒙特卡洛采样来估计

### 梯度计算的实现

In [ ]:
def compute_energy_gradient(wavefunction, samples, local_energies):
    """
    Compute the gradient of energy with respect to wave function parameters
    
    Parameters:
    wavefunction: neural network wave function
    samples: spin configurations, shape (batch_size, n_visible)
    local_energies: local energies for each sample, shape (batch_size,)
    
    Returns:
    gradient: dictionary of parameter gradients
    """
    # Ensure we're computing gradients
    wavefunction.zero_grad()
    
    # Compute log psi
    log_psi = wavefunction.log_psi(samples)
    
    # Compute gradients of log psi with respect to parameters
    log_psi.sum().backward(retain_graph=True)
    
    # Initialize gradient dictionary
    gradient = {}
    
    # Compute gradient for each parameter
    for name, param in wavefunction.named_parameters():
        if param.grad is not None:
            # Get gradient of log psi
            grad_log_psi = param.grad.detach()
            
            # Compute energy gradient
            # <E_L * O_k> - <E_L> * <O_k> where O_k = d(log psi)/d(theta_k)
            mean_EL = torch.mean(local_energies)
            mean_EL_Ok = torch.mean(local_energies.unsqueeze(1) * grad_log_psi, dim=0)
            mean_Ok = torch.mean(grad_log_psi, dim=0)
            
            energy_grad = 2 * (mean_EL_Ok - mean_EL * mean_Ok)
            
            gradient[name] = energy_grad
    
    return gradient

# Example: Compute energy gradient for a simple transverse field Ising model
def tfim_local_energy(samples, wavefunction, J=1.0, h=1.0):
    """
    Compute local energy for transverse field Ising model
    
    Parameters:
    samples: spin configurations, shape (batch_size, n_visible)
    wavefunction: neural network wave function
    J: coupling strength
    h: transverse field strength
    
    Returns:
    local_energies: local energies for each sample, shape (batch_size,)
    """
    batch_size, n_spins = samples.shape
    local_energies = torch.zeros(batch_size, device=samples.device)
    
    # Compute log psi for original samples
    log_psi_original = wavefunction.log_psi(samples)
    
    # ZZ interaction term
    for i in range(n_spins):
        j = (i + 1) % n_spins  # Periodic boundary conditions
        local_energies -= J * samples[:, i] * samples[:, j]
    
    # X field term (flip each spin and compute ratio)
    for i in range(n_spins):
        # Create samples with i-th spin flipped
        flipped_samples = samples.clone()
        flipped_samples[:, i] *= -1
        
        # Compute log psi for flipped samples
        log_psi_flipped = wavefunction.log_psi(flipped_samples)
        
        # Compute ratio
        ratio = torch.exp(log_psi_flipped - log_psi_original)
        
        # Add to local energy
        local_energies -= h * ratio
    
    return local_energies

# Compute local energies for our samples
local_energies = tfim_local_energy(spins, rbm)

# Compute energy gradient
energy_grad = compute_energy_gradient(rbm, spins, local_energies)

print("Energy gradients:")
for name, grad in energy_grad.items():
    print(f"{name}: {grad}")

# Compute average energy
mean_energy = torch.mean(local_energies)
print(f"\nAverage energy: {mean_energy.item()}")

## 3.3 随机梯度下降法

随机梯度下降法（Stochastic Gradient Descent, SGD）是优化神经网络量子态参数的基本方法。在变分蒙特卡洛方法中，我们使用SGD来最小化能量期望值。

### SGD的基本原理

SGD通过以下规则更新参数：

$$\theta_k \leftarrow \theta_k - \eta \frac{\partial E}{\partial \theta_k}$$

其中$\eta$是学习率，$\frac{\partial E}{\partial \theta_k}$是能量关于参数$\theta_k$的梯度。

在变分蒙特卡洛方法中，梯度是通过蒙特卡洛采样估计的：

$$\frac{\partial E}{\partial \theta_k} \approx 2 \left[ \frac{1}{N} \sum_{i=1}^{N} E_L(\mathbf{x}_i) O_k(\mathbf{x}_i) - \left(\frac{1}{N} \sum_{i=1}^{N} E_L(\mathbf{x}_i)\right) \left(\frac{1}{N} \sum_{i=1}^{N} O_k(\mathbf{x}_i)\right) \right]$$

其中$O_k(\mathbf{x}_i) = \frac{\partial \log \psi(\mathbf{x}_i)}{\partial \theta_k}$。

### SGD的特点

1. **简单高效**：算法简单，计算效率高
2. **随机性**：使用随机梯度估计，可以跳出局部极小值
3. **在线学习**：可以逐步更新参数，适合大规模问题
4. **超参数敏感**：对学习率等超参数的选择较为敏感

In [ ]:
def sgd_step(wavefunction, samples, learning_rate):
    """
    Perform one step of stochastic gradient descent
    
    Parameters:
    wavefunction: neural network wave function
    samples: spin configurations, shape (batch_size, n_visible)
    learning_rate: learning rate for SGD
    
    Returns:
    energy: average energy after the update
    """
    # Compute local energies
    local_energies = tfim_local_energy(samples, wavefunction)
    
    # Compute energy gradient
    energy_grad = compute_energy_gradient(wavefunction, samples, local_energies)
    
    # Update parameters
    with torch.no_grad():
        for name, param in wavefunction.named_parameters():
            if name in energy_grad:
                param -= learning_rate * energy_grad[name]
    
    # Return average energy
    return torch.mean(local_energies).item()

# Example: Perform SGD optimization
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Optimization parameters
n_epochs = 100
batch_size = 100
learning_rate = 0.01

# Track energy during optimization
energies = []

# Perform optimization
for epoch in range(n_epochs):
    # Generate random samples
    samples = torch.randint(0, 2, (batch_size, n_spins)).float() * 2 - 1
    samples = samples.to(device)
    
    # Perform SGD step
    energy = sgd_step(rbm, samples, learning_rate)
    energies.append(energy)
    
    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Energy: {energy:.6f}")

# Plot energy during optimization
plt.figure(figsize=(10, 6))
plt.plot(energies)
plt.xlabel('Epoch')
plt.ylabel('Energy')
plt.title('Energy Optimization with SGD')
plt.grid(True)
plt.show()

## 3.4 自适应优化方法

虽然基本的SGD方法简单有效，但在实际应用中，我们通常使用更先进的自适应优化方法，如Adam、RMSprop等。这些方法通过自适应地调整学习率，可以加速收敛并提高优化稳定性。

### Adam优化器

Adam（Adaptive Moment Estimation）是一种结合了动量法和自适应学习率的优化方法。它通过估计梯度的一阶矩和二阶矩来调整每个参数的学习率。

Adam的更新规则为：

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

其中$m_t$和$v_t$分别是梯度的一阶矩和二阶矩估计，$\beta_1$和$\beta_2$是矩估计的衰减率，$\eta$是学习率，$\epsilon$是一个小常数，用于数值稳定性。

### RMSprop优化器

RMSprop是另一种自适应学习率方法，它通过调整学习率来处理梯度中的不同尺度。

RMSprop的更新规则为：

$$E[g^2]_t = \beta E[g^2]_{t-1} + (1 - \beta) g_t^2$$
$$\theta_t = \theta_{t-1} - \eta \frac{g_t}{\sqrt{E[g^2]_t} + \epsilon}$$

其中$E[g^2]_t$是梯度平方的移动平均，$\beta$是衰减率，$\eta$是学习率，$\epsilon$是一个小常数。

### 自适应优化方法的特点

1. **自适应学习率**：根据参数的梯度历史自动调整学习率
2. **收敛速度快**：通常比基本SGD收敛更快
3. **稳定性高**：对学习率的选择不那么敏感
4. **内存开销大**：需要存储额外的状态信息

In [ ]:
def adam_step(wavefunction, samples, optimizer, t):
    """
    Perform one step of Adam optimization
    
    Parameters:
    wavefunction: neural network wave function
    samples: spin configurations, shape (batch_size, n_visible)
    optimizer: Adam optimizer
    t: current time step
    
    Returns:
    energy: average energy after the update
    """
    # Compute local energies
    local_energies = tfim_local_energy(samples, wavefunction)
    
    # Compute energy gradient
    energy_grad = compute_energy_gradient(wavefunction, samples, local_energies)
    
    # Update parameters using Adam
    optimizer.step(energy_grad, t)
    
    # Return average energy
    return torch.mean(local_energies).item()

class AdamOptimizer:
    """
    Adam optimizer for variational Monte Carlo
    
    Parameters:
    params: list of parameters to optimize
    lr: learning rate
    beta1: exponential decay rate for first moment estimates
    beta2: exponential decay rate for second moment estimates
    eps: small constant for numerical stability
    """
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        
        # Initialize moment estimates
        self.m = {name: torch.zeros_like(param) for name, param in self.params}
        self.v = {name: torch.zeros_like(param) for name, param in self.params}
        
    def step(self, grads, t):
        """
        Perform one optimization step
        
        Parameters:
        grads: dictionary of parameter gradients
        t: current time step
        """
        with torch.no_grad():
            for (name, param), m, v in zip(self.params, self.m.values(), self.v.values()):
                if name in grads:
                    grad = grads[name]
                    
                    # Update biased first moment estimate
                    m[:] = self.beta1 * m + (1 - self.beta1) * grad
                    
                    # Update biased second moment estimate
                    v[:] = self.beta2 * v + (1 - self.beta2) * grad * grad
                    
                    # Compute bias-corrected first moment estimate
                    m_hat = m / (1 - self.beta1 ** t)
                    
                    # Compute bias-corrected second moment estimate
                    v_hat = v / (1 - self.beta2 ** t)
                    
                    # Update parameters
                    param -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

# Example: Perform Adam optimization
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Create Adam optimizer
params = [(name, param) for name, param in rbm.named_parameters()]
optimizer = AdamOptimizer(params, lr=0.01)

# Optimization parameters
n_epochs = 100
batch_size = 100

# Track energy during optimization
energies_adam = []

# Perform optimization
for epoch in range(n_epochs):
    # Generate random samples
    samples = torch.randint(0, 2, (batch_size, n_spins)).float() * 2 - 1
    samples = samples.to(device)
    
    # Perform Adam step
    energy = adam_step(rbm, samples, optimizer, epoch + 1)
    energies_adam.append(energy)
    
    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Energy: {energy:.6f}")

# Compare SGD and Adam
plt.figure(figsize=(10, 6))
plt.plot(energies, label='SGD')
plt.plot(energies_adam, label='Adam')
plt.xlabel('Epoch')
plt.ylabel('Energy')
plt.title('Energy Optimization: SGD vs Adam')
plt.legend()
plt.grid(True)
plt.show()

## 3.5 自然梯度法

自然梯度法（Natural Gradient Descent, NGD）是一种考虑参数空间几何结构的优化方法。在变分蒙特卡洛方法中，自然梯度法可以显著提高优化效率，特别是在处理复杂波函数时。

### 自然梯度的基本原理

自然梯度考虑了参数空间的黎曼几何结构，通过费雪信息矩阵（Fisher Information Matrix）来调整梯度方向：

$$\tilde{\nabla}E = F^{-1} \nabla E$$

其中$\nabla E$是常规梯度，$F$是费雪信息矩阵，$\tilde{\nabla}E$是自然梯度。

费雪信息矩阵的元素为：

$$F_{ij} = \langle O_i O_j \rangle - \langle O_i \rangle \langle O_j \rangle$$

其中$O_i = \frac{\partial \log \psi}{\partial \theta_i}$是波函数对数关于参数的梯度。

### 自然梯度的优势

1. **几何感知**：考虑了参数空间的几何结构
2. **参数无关**：对参数的重新参数化不敏感
3. **收敛速度快**：通常比常规梯度下降收敛更快
4. **稳定性高**：对学习率的选择不那么敏感

### 自然梯度的计算挑战

1. **矩阵求逆**：需要计算费雪信息矩阵的逆，计算复杂度高
2. **矩阵病态**：费雪信息矩阵可能病态，导致数值不稳定
3. **内存消耗**：存储费雪信息矩阵需要大量内存

为了解决这些挑战，实际应用中通常使用近似方法，如共轭梯度法、随机自然梯度法等。

In [ ]:
def compute_fisher_matrix(wavefunction, samples):
    """
    Compute Fisher information matrix
    
    Parameters:
    wavefunction: neural network wave function
    samples: spin configurations, shape (batch_size, n_visible)
    
    Returns:
    F: Fisher information matrix
    param_names: list of parameter names
    """
    # Ensure we're computing gradients
    wavefunction.zero_grad()
    
    # Compute log psi
    log_psi = wavefunction.log_psi(samples)
    
    # Compute gradients of log psi with respect to parameters
    log_psi.sum().backward(retain_graph=True)
    
    # Collect parameter gradients
    param_names = []
    grads = []
    
    for name, param in wavefunction.named_parameters():
        if param.grad is not None:
            param_names.append(name)
            # Flatten the gradient
            grads.append(param.grad.detach().flatten())
    
    # Concatenate all gradients
    O = torch.cat(grads, dim=0)  # Shape: (batch_size, n_params)
    
    # Compute Fisher information matrix
    # F_ij = <O_i O_j> - <O_i><O_j>
    O_mean = torch.mean(O, dim=0)  # Shape: (n_params,)
    F = torch.matmul(O.T, O) / O.size(0) - torch.outer(O_mean, O_mean)  # Shape: (n_params, n_params)
    
    return F, param_names

def natural_gradient_step(wavefunction, samples, learning_rate, damping=1e-3):
    """
    Perform one step of natural gradient descent
    
    Parameters:
    wavefunction: neural network wave function
    samples: spin configurations, shape (batch_size, n_visible)
    learning_rate: learning rate
    damping: damping parameter for numerical stability
    
    Returns:
    energy: average energy after the update
    """
    # Compute local energies
    local_energies = tfim_local_energy(samples, wavefunction)
    
    # Compute energy gradient
    energy_grad = compute_energy_gradient(wavefunction, samples, local_energies)
    
    # Compute Fisher information matrix
    F, param_names = compute_fisher_matrix(wavefunction, samples)
    
    # Add damping for numerical stability
    F += damping * torch.eye(F.size(0), device=F.device)
    
    # Collect energy gradients into a vector
    grad_vector = []
    for name in param_names:
        if name in energy_grad:
            grad_vector.append(energy_grad[name].flatten())
    
    grad_vector = torch.cat(grad_vector, dim=0)  # Shape: (n_params,)
    
    # Compute natural gradient: F^{-1} * grad
    try:
        nat_grad_vector = torch.linalg.solve(F, grad_vector)
    except torch.linalg.LinAlgError:
        # If matrix is singular, use pseudo-inverse
        nat_grad_vector = torch.linalg.pinv(F) @ grad_vector
    
    # Update parameters
    with torch.no_grad():
        idx = 0
        for name, param in wavefunction.named_parameters():
            if name in energy_grad:
                param_shape = param.shape
                param_size = param.numel()
                
                # Extract natural gradient for this parameter
                nat_grad = nat_grad_vector[idx:idx+param_size].reshape(param_shape)
                idx += param_size
                
                # Update parameter
                param -= learning_rate * nat_grad
    
    # Return average energy
    return torch.mean(local_energies).item()

# Example: Perform natural gradient optimization
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Optimization parameters
n_epochs = 100
batch_size = 100
learning_rate = 0.01

# Track energy during optimization
energies_ngd = []

# Perform optimization
for epoch in range(n_epochs):
    # Generate random samples
    samples = torch.randint(0, 2, (batch_size, n_spins)).float() * 2 - 1
    samples = samples.to(device)
    
    # Perform natural gradient step
    energy = natural_gradient_step(rbm, samples, learning_rate)
    energies_ngd.append(energy)
    
    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Energy: {energy:.6f}")

# Compare SGD, Adam, and Natural Gradient
plt.figure(figsize=(10, 6))
plt.plot(energies, label='SGD')
plt.plot(energies_adam, label='Adam')
plt.plot(energies_ngd, label='Natural Gradient')
plt.xlabel('Epoch')
plt.ylabel('Energy')
plt.title('Energy Optimization: SGD vs Adam vs Natural Gradient')
plt.legend()
plt.grid(True)
plt.show()

## 总结

本节介绍了变分蒙特卡洛方法中的优化技术，包括：

1. **变分原理**：介绍了变分原理的数学表述、物理意义以及在蒙特卡洛方法中的应用

2. **能量梯度计算**：详细介绍了能量梯度的数学表达式、物理意义和实现方法

3. **随机梯度下降法**：介绍了SGD的基本原理、特点和实现

4. **自适应优化方法**：介绍了Adam和RMSprop等自适应优化方法的原理和特点

5. **自然梯度法**：介绍了自然梯度的基本原理、优势、计算挑战和实现方法

这些优化方法是变分蒙特卡洛方法的核心技术，它们决定了神经网络量子态参数优化的效率和稳定性。在实际应用中，我们需要根据具体问题的特点选择合适的优化方法。